In [1]:
import pygraphviz as pgv

class Node:
    def __init__(self, parent=None, children=None, data=None, tag=None):
        """
        结点数据结构
        :param parent:  父节点
        :param children: 子节点，列表结构
        :param data: 数据域
        """
        if children is None:
            children = []
        self.tag = tag
        self.parent = parent
        self.data = data
        self.children = children
    
    def __str__(self):
        return self.tag

class Tree:
    def __init__(self, rootdata):
        self.root = Node(data=rootdata, tag='root')

    def insert(self, parent_node, children_node):
        children_node.parent = parent_node
        parent_node.children.append(children_node)

    def search(self, node, data):
        """
        以node为根节点查找值为data的结点，返回
        :param node: 根节点
        :param tag: 标签
        :return:
        """
        if node.data == data:
            return node
        elif len(node.children) == 0:
            return None
        else:
            for child in node.children:
                res = self.search(child, data)
                if res is not None:
                    return res
            return None

    def get_leavesByDataRoute(self, data_list):
        """
        根据数据域提供的列表查找该路径下的所有叶子集合
        :param data_list: 路径的数据域列表，[子节点数据， ……]
        :return: 该路径下的叶子数据集合
        """
        leaves = set()
        node = self.root
        for data in data_list:
            node = self.search(node, data)

        for child in node.children:
            if len(child.children) > 0:
                leaves.add(child.data)

        return leaves

    def show(self, save_path=None):
        """
        显示该树形结构
        :return:
        """
        nest_format = '\<{:s}\>'
        node_shape = 'point'
        node_style = 'dashed'
        node_color = 'blue'
        dg = pgv.AGraph(directed=True, rankdir='TB')

        def print_node(node):
            if len(node.children) > 0:
                for child in node.children:
                    # dg.add_node(child.tag, label=nest_format.format(child.data), shape=node_shape, \
                    #         style=node_style, color=node_color, fontname="Sans", fontsize='10')
                    dg.add_node(child.tag, shape=node_shape, \
                            style=node_style, color=node_color, fontname="Sans", fontsize='10')
                    dg.add_edge(node.tag, child.tag, arrowsize='0.2')# 只能用string类型输入参数
                    print_node(child)

        # dg.add_node(self.root.tag, label=self.root.data, shape=node_shape, style=node_style, color=node_color)
        dg.add_node(self.root.tag, shape=node_shape, style=node_style, color=node_color)
        print_node(self.root)
        # dg.layout(prog='dot')
        # dg.draw('D:\\OR\\current\\draft\\要添加的东西\\label.pdf', prog='dot')
        dg.draw('test.pdf', prog='dot')
        if save_path is not None:
            dg.render(save_path)


In [2]:
def manual_tree():
    tree = Tree('root')
    root = tree.root

    tag_list = ['1', '2', '0', 
                '1,1', '1,2', '2,1', '2,2', 
                '1,1,1', '1,1,2', '1,1,3', 
                '1,2,1', '1,2,2', '1,2,3', 
                '2,1,1', '2,1,2', '2,1,3', 
                '2,2,1', '2,2,2', '2,2,3', ]

    node_dict = {}
    node_dict['root'] = root
    for k in tag_list:
        node_dict[k] = Node(tag=k, data={})
    
    # level 1
    tree.insert(root, node_dict['1'])

    tree.insert(root, node_dict['2'])

    tree.insert(root, node_dict['0'])

    # level 2
    tree.insert(node_dict['1'], node_dict['1,1'])
    tree.insert(node_dict['1'], node_dict['1,2'])

    tree.insert(node_dict['2'], node_dict['2,1'])
    tree.insert(node_dict['2'], node_dict['2,2'])
    
    # level 3
    for i in range(1, 4):
        tree.insert(node_dict['1,1'], node_dict['1,1,'+str(i)])

    for i in range(1, 4):
        tree.insert(node_dict['1,2'], node_dict['1,2,'+str(i)])

    for i in range(1, 4):
        tree.insert(node_dict['2,1'], node_dict['2,1,'+str(i)])

    for i in range(1, 4):
        tree.insert(node_dict['2,2'], node_dict['2,2,'+str(i)])

    return tree, node_dict



In [3]:
# 设定参数全局变量、对position bias离散化
import numpy as np

mlnl, NODE = manual_tree() # mlnl只是一个tree
# mlnl.show() # 文件输出到test.pdf中

# 设置scale parameter，一共6个，root的固定为1
MU = {}

MU['root'] = 1

MU['1'] = 2
MU['2'] = 1.5

MU['1,1'] = 4
MU['1,2'] = 4
MU['2,1'] = 3
MU['2,2'] = 3

# 设置exponential utility，一共12个

NODE['1,1,1'].data['a'] = 3.5
NODE['1,1,2'].data['a'] = 2.6
NODE['1,1,3'].data['a'] = 2.4

NODE['1,2,1'].data['a'] = 4.0
NODE['1,2,2'].data['a'] = 3.8
NODE['1,2,3'].data['a'] = 2.5

NODE['2,1,1'].data['a'] = 3.9
NODE['2,1,2'].data['a'] = 3.0
NODE['2,1,3'].data['a'] = 2.7

NODE['2,2,1'].data['a'] = 4.1
NODE['2,2,2'].data['a'] = 2.9
NODE['2,2,3'].data['a'] = 2.8

GAMMA = [0.50, 0.53, 0.75, 0.86, 1.1, 1.2]
OMEGA = [1] * 6
def bias_discretize(beta, omega, rho=0.2):
    '''
    函数说明：用于将beta和omega离散化到给定mu_0,rho的刻度区间内
    参数说明：beta是原有的exp(position bias)，omega是每个值的个数，m是区间的个数
    实现方式：直接根据m的值做一个数值频数统计即可
    '''
    # b2 = max(GAMMA)
    b1 = min(GAMMA)
    ratio = np.power(1+rho, 1/MU['root'])
    i = 0
    j = 0
    g = [] # 有效区间的左端点
    o = {} # 每个有效区间的位置数目
    while j < len(beta):
        if beta[j] < b1 * np.power(ratio, i+1):
            k = b1 * np.power(ratio, i)
            if not k in o.keys():
                o[k] = omega[j]
                g.append(k)
            else:
                o[k] += omega[j]
            j += 1
        else:
            i += 1
    return g, [o[k] for k in g]


GAMMA, OMEGA = bias_discretize(GAMMA, OMEGA)
# print(GAMMA, OMEGA)



In [12]:
# # 单独一个离散化的例子
# GAMMA = [0.50, 0.53, 0.75, 0.86, 1.1, 1.2, 1.5]
# OMEGA = [1] * 7
# print(np.log(GAMMA))
# GAMMA, OMEGA = bias_discretize(GAMMA, OMEGA)
# print(GAMMA, OMEGA)

# print(np.log(GAMMA))

[-0.69314718 -0.63487827 -0.28768207 -0.15082289  0.09531018  0.18232156
  0.40546511]
[0.5, 0.72, 1.0368, 1.4929919999999997] [2, 2, 2, 1]
[-0.69314718 -0.32850407  0.03613905  0.40078216]


In [11]:
# DP用到的函数，以及打表查表用到全局变量
from copy import deepcopy


S = {} # 这个在论文里是S_{i_1,i_2,...,i_l}(\omega)
J_KEY = {} # 这个可不是J的值，而是J取得这个值所用的omega，
# 在论文里是\omega_{i_1,i_2,...,i_l}(\omega)
J_VALUE = {}

def pow_set(s):
    '''
    函数说明：给定一个list，返回它的power set
    '''
    l = len(s)
    tmp = [[j for j in range(s[i]+1)] for i in range(l)]
    
    r = []
    for i in range(l):
        t = tmp[i]
        if len(r) == 0:
            r = [[j] for j in t]
        else:
            newr = []
            for j in t:
                for k in r:
                    m = deepcopy(k)
                    m.extend([j])
                    newr.append(m)
            r = deepcopy(newr)
        
    return r      


def diff_set(a, b):
    '''
    函数说明：返回a-b
    '''
    return [a[i]-b[i] for i in range(len(a))]

def exp_mu_z(p, s, node=NODE):
    '''
    函数说明：用于计算h'_{i_1,i_2,...,i_l}(S)
    参数说明：p是h的下标，s是S，即assortment-position solution
    '''
    pos = p.split(',')
    if len(pos) == 3: # 3是3层的意思
        if p in s.keys():
            # s作为一个solution就是一个指派
            return np.power(node[p].data['a'] * GAMMA[s[p]], MU['root'])
        else:
            return 0
    else:
        tmp = 0
        for child in node[p].children:
            tmp += np.power(exp_mu_z(child.tag, s), MU[p]/MU['root'])
        return np.power(tmp, MU['root'] / MU[p])

def compute_s(p, omega=OMEGA, node=NODE):
    '''
    函数说明：用于计算S_{i_1,i_2,...,i_l}(\omega)
    '''
    o = str(omega) # 做这一步是因为omega是个list，unhashable
    if (p, o) in S.keys():
        return S[p, o]
    if sum(omega) == 0:
        S[p, o] = {}
        return S[p, o]
    
    s = {}
    pos = p.split(',')
    if len(pos) == 3: # 3是3层的意思
        i = len(omega)
        for j in omega[::-1]:
            i -= 1
            if not j == 0:
                s[p] = i
                break
    else:
        # 如何把J_KEY里的信息读过来
        # 然后把这种信息解析为S_KEY
        om = [0] * len(omega)
        ds = omega
        for child in node[p].children:
            ds = diff_set(ds, om)
            key = (child.tag, str(ds))
            if not key in J_KEY.keys():
                compute_j(child.tag, ds)
            om = J_KEY[key]
            # if p == '1,1':
            #     print(child.tag, om)
            # s = {**s, **compute_s(child.tag, om)}
            s = {**s, **S[child.tag, str(om)]}

    S[p, o] = s
    return S[p, o]

def compute_s_root(p='root', omega=OMEGA, node=NODE):
    '''
    函数说明：用于计算S_{i_1,i_2,...,i_l}(\omega'_0)
    '''
    
    s = {}
    pos = p.split(',')
    om = [0] * len(omega)
    ds = omega
    for child in node[p].children[:-1]:
        ds = diff_set(ds, om)
        key = (child.tag, str(ds))
        if not key in J_KEY.keys():
            compute_j(child.tag, ds)
        om = J_KEY[key]
        # if p == '1,1':
        #     print(child.tag, om)
        # s = {**s, **compute_s(child.tag, om)}
        s = {**s, **S[child.tag, str(om)]}

    return s

def compute_j(p, omega=OMEGA, node=NODE):
    '''
    函数说明：用于计算J_{i_1,i_2,...,i_l}(\omega)
    参数说明：p是J的下标，omega是position arrangement，默认为全集
    '''
    # flag = (p == '1')
    if p not in node.keys():
        return 0
    
    o = str(omega) # 做这一步是因为omega是个list，unhashable
    if (p, o) in J_VALUE.keys():
        return J_VALUE[p, o]

    p_s = pow_set(omega)
    
    p_sibling = p[:-1]+str(int(p[-1])+1)
    max_value = -1
    for ps in p_s:
        compute_s(p, ps)
        v = exp_mu_z(p, S[p, str(ps)]) + compute_j(p_sibling, diff_set(omega, ps))
        if v > max_value:
            # if flag:
            #     print(p, ps)
            #     print(v)
            max_key = ps
            max_value = v
    
    J_KEY[p, o] = max_key
    J_VALUE[p, o] = max_value

    return max_value



In [12]:
# 动态规划主程序，输出：字典，key是商品编号，value是position的编号


nest_list = ['1', '2', 
            '1,1', '1,2', '2,1', '2,2', 
            '1,1,1', '1,1,2', '1,1,3', 
            '1,2,1', '1,2,2', '1,2,3', 
            '2,1,1', '2,1,2', '2,1,3', 
            '2,2,1', '2,2,2', '2,2,3', ]

# 改成根据随机种子自动生成，包括节点及其引用和参数设置等

for i in nest_list[::-1]:
    for ps in pow_set(OMEGA):
        compute_j(i, ps)




# print(J_VALUE['1,1,1', str([0,0,1])])
# omega111 = J_KEY['1,1', str([0,0,1])]
# print(omega111)
# print(J_VALUE['1,1,1', str([0,0,1])])
# print(S['1,1,1', str(omega111)])
# print(exp_mu_z('1,1,3', {'1,1,3':2}))
# print(S['1', str(OMEGA)])
# print(J_VALUE['1', str(OMEGA)])
# print(J_KEY['1,1,1', '[1, 0, 0]'])
# print()
# print(GAMMA)
# print(OMEGA)

# s_discrete = compute_s_root() # 需要到前面把GAMMA, OMEGA换成discretize的
s_original = compute_s_root()

# print(s_discrete)
# {'1,1,1': 1, '1,2,1': 2, '2,1,1': 1, '2,1,2': 0, '2,1,3': 0, '2,2,1': 2}
print(s_original)
# {'1,1,1': 2, '1,2,1': 5, '2,1,1': 3, '2,1,2': 1, '2,1,3': 0, '2,2,1': 4}

{'1,1,1': 2, '1,2,1': 5, '2,1,1': 3, '2,1,2': 1, '2,1,3': 0, '2,2,1': 4}


In [ ]:
omega_list = [[1, 0, 0], [0, 1, 0], [1, 1, 0], [], [0, 2, 2], [1, 2, 2], [2, 2, 2]]
f = open("中间变量输出.txt", "w")
counter = 1
for i in nest_list[::-1]:
    if len(i) == 5: # 叶节点
        print("({}) & ".format(counter), end="", file=f)
        counter += 1
        print("$\\langle"+i, end="", file=f)
        print("\\rangle$", end=" & ", file=f)
        # 在这里输出psi
        print("$h'_{"+i+"}(S_{"+i+"}(\\omega))$", end=" & ",file=f)
        for o in omega_list:
            if len(o) == 0:
                print("...", end=" & ", file=f)
            else:
                psi = exp_mu_z(i, S[i, str(o)])
                print("{:.2f}".format(psi), end=" & ", file=f)
        print("\\\\", file=f)

    else:
        print("({}) & ".format(counter), end="", file=f)
        counter += 1
        print("$\\langle"+i, end="", file=f)
        print("\\rangle$", end=" & ", file=f)
        # 在这里输出psi
        print("$h'_{"+i+"}(S_{"+i+"}(\\omega))$", end=" & ",file=f)
        for o in omega_list:
            if len(o) == 0:
                print("...", end=" & ", file=f)
            else:
                psi = exp_mu_z(i, S[i, str(o)])
                print("{:.2f}".format(psi), end=" & ", file=f)
        print("\\\\", file=f)
        print("({}) & ".format(counter), end="", file=f)
        counter += 1
        print("~", end=" & ", file=f) # 第一列空着
        # 在这里输出omega^*_{}(omega)
        print("$\\omega^*_{"+i+"}(\\omega)$", end=" & ", file=f)
        for o in omega_list:
            if len(o) == 0:
                print("...", end=" & ", file=f)
            else: 
                p = J_KEY[i, str(o)]
                print("({},{},{})".format(p[0],p[1],p[2]), end=" & ", file=f)
        print("\\\\", file=f)
        print("({}) & ".format(counter), end="", file=f)
        counter += 1
        print("~", end=" & ", file=f) # 第一列空着
        # 在这里输出J_VALUE
        print("$J_{"+i+"}(\\omega)$", end=" & ", file=f)
        for o in omega_list:
            if len(o) == 0:
                print("...", end=" & ", file=f)
            else: 
                j = J_VALUE[i, str(o)]
                print("{:.2f}".format(j), end=" & ", file=f)
        print("\\\\", file=f)
    
f.close()